In [1]:
import pandas as pd
from database import postgres_connection


In [2]:
def load_data():
    conn = postgres_connection()

    try:
        user_df = pd.read_sql(
            """
            SELECT 
            * 
            FROM public.user_data
            """,
            conn,
        )

        post_df = pd.read_sql(
            """
            SELECT 
            *
            FROM public.post_text_df 
            """,
            conn,
        )

        feed_df = pd.read_sql(
            """
            SELECT 
            *
            FROM public.feed_data
            LIMIT 5000000
            """,
            conn,
        )

    except Exception as e:
        print(f"Ошибка запроса: {e}")
        raise

    finally:
        if conn:
            conn.close()

    return user_df, post_df, feed_df


# user_df, post_df, feed_df = load_data()

In [3]:
# print(user_df.shape)
# print(post_df.shape)
# print(feed_df.shape)

In [4]:
# print(user_df.head(5))
# print("=" * 100)
# print(post_df.head(5))
# print("=" * 100)
# print(feed_df.head(5))

In [5]:
# Сохранение данных в data/ чтобы в случае перезапуска каждый раз не загружать
# tabs_dict = {"user_df": user_df, "post_df": post_df, "feed_df": feed_df}

# for name, df in tabs_dict.items():
#     df.to_csv(f"data/{name}_{df.shape[0]}rows.csv", index=False)

In [6]:
user_df = pd.read_csv("data/user_df_163205rows.csv", sep=",")
post_df = pd.read_csv("data/post_df_7023rows.csv", sep=",")
feed_df = pd.read_csv("data/feed_df_5000000rows.csv", sep=",")

In [7]:
print(user_df.shape)
print(post_df.shape)
print(feed_df.shape)

(163205, 8)
(7023, 3)
(5000000, 5)


In [8]:
user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163205 entries, 0 to 163204
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   user_id    163205 non-null  int64 
 1   gender     163205 non-null  int64 
 2   age        163205 non-null  int64 
 3   country    163205 non-null  object
 4   city       163205 non-null  object
 5   exp_group  163205 non-null  int64 
 6   os         163205 non-null  object
 7   source     163205 non-null  object
dtypes: int64(4), object(4)
memory usage: 10.0+ MB


In [9]:
user_df.isna().sum()

user_id      0
gender       0
age          0
country      0
city         0
exp_group    0
os           0
source       0
dtype: int64

In [10]:
user_df["user_id"].is_unique

True

In [11]:
post_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7023 entries, 0 to 7022
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   post_id  7023 non-null   int64 
 1   text     7023 non-null   object
 2   topic    7023 non-null   object
dtypes: int64(1), object(2)
memory usage: 164.7+ KB


In [12]:
post_df.isna().sum()

post_id    0
text       0
topic      0
dtype: int64

In [13]:
post_df["post_id"].is_unique

True

In [14]:
feed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 5 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   timestamp  object
 1   user_id    int64 
 2   post_id    int64 
 3   action     object
 4   target     int64 
dtypes: int64(3), object(2)
memory usage: 190.7+ MB


In [15]:
feed_df.describe(include="all")

,timestamp,user_id,post_id,action,target
count,5000000,5.000000e+06,5.000000e+06,5000000,5.000000e+06
unique,800492,NaN,NaN,2,NaN
top,2021-12-10 13:32:02,NaN,NaN,view,NaN
freq,44,NaN,NaN,4468243,NaN
mean,NaN,8.788183e+04,3.396350e+03,NaN,1.063520e-01
std,NaN,4.935450e+04,2.095470e+03,NaN,3.082876e-01
min,NaN,4.760000e+02,1.000000e+00,NaN,0.000000e+00
25%,NaN,4.142100e+04,1.527000e+03,NaN,0.000000e+00
50%,NaN,9.954500e+04,3.192000e+03,NaN,0.000000e+00
75%,NaN,1.283450e+05,5.205000e+03,NaN,0.000000e+00


In [16]:
feed_df.target.value_counts(dropna=False)

target
0    4468240
1     531760
Name: count, dtype: int64

In [17]:
feed_df.action.value_counts(dropna=False)

action
view    4468243
like     531757
Name: count, dtype: int64

In [18]:
feed_df.isna().sum()

timestamp    0
user_id      0
post_id      0
action       0
target       0
dtype: int64

In [19]:
pd.crosstab(
    feed_df["action"],
    feed_df["target"].fillna("missing"),
)

target,0,1
action,,
like,531757,0
view,3936483,531760


In [20]:
pd.crosstab(
    feed_df["action"],
    feed_df["target"].fillna("missing"),
    margins=True,
    normalize="index",
)

target,0,1
action,,
like,1.000000,0.000000
view,0.880991,0.119009
All,0.893648,0.106352


In [21]:
feed_df.target.isna().sum()

np.int64(0)

In [22]:
feed_df.loc[feed_df["action"] == "like", "target"].value_counts(dropna=False)

target
0    531757
Name: count, dtype: int64

In [23]:
feed_df[feed_df["action"] == "like"].head(5)

,timestamp,user_id,post_id,action,target
1,2021-11-16 19:06:34,476,3873,like,0
44,2021-11-18 18:04:15,476,707,like,0
64,2021-11-25 13:52:20,476,5155,like,0
86,2021-11-25 14:28:59,476,6210,like,0
96,2021-12-01 10:07:07,476,383,like,0


In [24]:
feed_df.groupby("action")[["user_id", "post_id"]].value_counts()

action  user_id  post_id
like    142167   1175       5
        544      1426       3
        647      3833       3
        671      1423       3
        799      6742       3
                           ..
view    161883   7199       1
                 7257       1
                 7261       1
                 7296       1
                 7311       1
Name: count, Length: 4787927, dtype: int64

In [25]:
user_id, post_id = feed_df[feed_df["action"] == "like"][["user_id", "post_id"]].iloc[0]

feed_df[(feed_df["user_id"] == user_id) & (feed_df["post_id"] == post_id)]

,timestamp,user_id,post_id,action,target
0,2021-11-16 19:05:49,476,3873,view,1
1,2021-11-16 19:06:34,476,3873,like,0


В таблице feed_data событие лайка хранится отдельно от события просмотра. Для положительного взаимодействия сначала присутствует строка view с target=1, затем отдельная строка like с техническим значением target=0. Поэтому для формирования обучающей выборки используются только события view, а строки like исключаются, чтобы не создавать ложные отрицательные примеры.

In [26]:
view_df = feed_df[feed_df["action"] == "view"]
print(view_df.shape[0])
print(view_df.target.value_counts(dropna=False))
neg, pos = view_df.target.value_counts(dropna=False)
print(f"{pos / (neg + pos) * 100:.2f}%")

4468243
target
0    3936483
1     531760
Name: count, dtype: int64
11.90%


In [27]:
print(feed_df.user_id.nunique())
print(feed_df.post_id.nunique())
print(feed_df.timestamp.min())
print(feed_df.timestamp.max())

10636
6831
2021-10-01 06:01:40
2021-12-29 23:43:15


In [28]:
feed_df.groupby("user_id")["action"].count().sort_values(ascending=False).head(3)

user_id
142343    936
52140     925
1049      909
Name: action, dtype: int64

In [29]:
# import os

# os.makedirs("data", exist_ok=True)

In [30]:
feed_df.head(5)

,timestamp,user_id,post_id,action,target
0,2021-11-16 19:05:49,476,3873,view,1
1,2021-11-16 19:06:34,476,3873,like,0
2,2021-11-16 19:06:36,476,720,view,0
3,2021-11-16 19:07:51,476,578,view,0
4,2021-11-16 19:10:17,476,1234,view,0


# Создание признаков

In [31]:
print(user_df.head(5))
print("=" * 100)
print(post_df.head(5))
print("=" * 100)
print(feed_df.head(5))

   user_id  gender  age country               city  exp_group       os source
0      200       1   34  Russia          Degtyarsk          3  Android    ads
1      201       0   37  Russia             Abakan          0  Android    ads
2      202       1   17  Russia           Smolensk          4  Android    ads
3      203       0   18  Russia             Moscow          1      iOS    ads
4      204       0   36  Russia  Anzhero-Sudzhensk          3  Android    ads
   post_id                                               text     topic
0        1  UK economy facing major risks\n\nThe UK manufa...  business
1        2  Aids and climate top Davos agenda\n\nClimate c...  business
2        3  Asian quake hits European shares\n\nShares in ...  business
3        4  India power shares jump on debut\n\nShares in ...  business
4        5  Lacroix label bought by US firm\n\nLuxury good...  business
             timestamp  user_id  post_id action  target
0  2021-11-16 19:05:49      476     3873   v

In [32]:
print(post_df.shape)
print(post_df.post_id.nunique())
print(post_df.text.nunique())

(7023, 3)
7023
6924


### Очистка и предобработка текстов постов


In [33]:
import re
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("en_core_web_sm")

# 1. создаем копию датафрейма с уникальными текстами постов
# unique_texts_df = post_df.drop_duplicates(subset="text", keep="first")
# print(unique_texts_df.shape)


# 2. определяем функцию для предобработки текста
def pre_clean(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)  # Удаляем ссылки
    text = re.sub(r"[^a-z\s]", "", text)  # Оставляем только латиницу и пробелы

    return text


# применяем первичную очистку текста
post_df["pre_cleaned_text"] = post_df["text"].apply(pre_clean)

# 3. УСКОРЕННАЯ ЛЕММАТИЗАЦИЯ ЧЕРЕЗ nlp.pipe
# Отключаем лишнее (ner, parser)
cleaned_texts = []
docs = nlp.pipe(
    post_df["pre_cleaned_text"].astype("str"),
    disable=["ner", "parser"],
)

for doc in docs:
    # Убираем английские стоп-слова и берем лемму (начальную форму)
    lemma = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    cleaned_texts.append(" ".join(lemma))

# Записываем результат в датафрейм
post_df["cleaned_text"] = cleaned_texts


In [34]:
print(post_df.shape)
# print("--- Очищенный текст для модели ---")
# print(unique_texts_df[["text", "cleaned_text"]], "\n")


(7023, 5)


In [36]:
# 4. СВЯЗКА С TF-IDF
# max_features=1000 ограничит словарь до 50 самых важных слов, чтобы не перегружать память
tfidf = TfidfVectorizer(max_features=50)

# Обучаем TF-IDF на ускоренно очищенной колонке
tfidf_matrix = tfidf.fit_transform(post_df["cleaned_text"])
feature_names = tfidf.get_feature_names_out()
# Переводим в DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names).add_prefix(
    "tfidf_"
)
print(tfidf_df.shape)

(7023, 50)


In [38]:
# Сбрасываем индексы исходного df, чтобы они шли строго от 0 до конца без пропусков
# unique_texts_df = unique_texts_df.reset_index(drop=True)

# 5. Итоговый датасет: метаданные + векторные признаки
post_tfidf_processed_df = pd.concat([post_df[["post_id", "topic"]], tfidf_df], axis=1)
print(post_tfidf_processed_df.shape)


(7023, 52)


In [39]:
float_columns = post_tfidf_processed_df.select_dtypes(include=["float64"]).columns
post_tfidf_processed_df[float_columns] = post_tfidf_processed_df[float_columns].astype(
    "float32"
)

In [40]:
post_tfidf_processed_df.shape

(7023, 52)

### Оставляем только действия с action == view

In [41]:
feed_view_df = feed_df[feed_df["action"] == "view"]

print(feed_view_df.shape)
print(feed_view_df.head())


(4468243, 5)
             timestamp  user_id  post_id action  target
0  2021-11-16 19:05:49      476     3873   view       1
2  2021-11-16 19:06:36      476      720   view       0
3  2021-11-16 19:07:51      476      578   view       0
4  2021-11-16 19:10:17      476     1234   view       0
5  2021-11-16 19:13:15      476     1762   view       0


### Добавляем временные признаки: hour, day_of_week, month

In [42]:
from datetime import datetime as dt

# print(dt.now())

datetime_df = feed_view_df.timestamp  # .reset_index(drop=True)
print(datetime_df.head(5))
print(datetime_df.dtypes)


def hour_dow_mnth_from_timestamp(datetime_df):
    if not pd.api.types.is_datetime64_any_dtype(datetime_df):
        datetime_df = pd.to_datetime(datetime_df)
    temp_df = pd.DataFrame()
    temp_df["hour"] = datetime_df.dt.hour
    temp_df["dow"] = datetime_df.dt.weekday
    temp_df["month"] = datetime_df.dt.month

    return temp_df


date_df = hour_dow_mnth_from_timestamp(datetime_df)

print("=" * 100)
print(date_df.head(5))

0    2021-11-16 19:05:49
2    2021-11-16 19:06:36
3    2021-11-16 19:07:51
4    2021-11-16 19:10:17
5    2021-11-16 19:13:15
Name: timestamp, dtype: object
object
   hour  dow  month
0    19    1     11
2    19    1     11
3    19    1     11
4    19    1     11
5    19    1     11


In [43]:
print(feed_view_df.shape)
print(date_df.shape)

feed_view_df = pd.concat([feed_view_df, date_df], axis=1)

(4468243, 5)
(4468243, 3)


In [44]:
feed_view_df.head()

,timestamp,user_id,post_id,action,target,hour,dow,month
0,2021-11-16 19:05:49,476,3873,view,1,19,1,11
2,2021-11-16 19:06:36,476,720,view,0,19,1,11
3,2021-11-16 19:07:51,476,578,view,0,19,1,11
4,2021-11-16 19:10:17,476,1234,view,0,19,1,11
5,2021-11-16 19:13:15,476,1762,view,0,19,1,11


In [45]:
learning_table = pd.merge(feed_view_df, user_df, on="user_id", how="left")
print(learning_table.shape)
learning_table.head()

(4468243, 15)


,timestamp,user_id,post_id,action,target,hour,dow,month,gender,age,country,city,exp_group,os,source
0,2021-11-16 19:05:49,476,3873,view,1,19,1,11,1,23,Russia,Izhevsk,3,Android,ads
1,2021-11-16 19:06:36,476,720,view,0,19,1,11,1,23,Russia,Izhevsk,3,Android,ads
2,2021-11-16 19:07:51,476,578,view,0,19,1,11,1,23,Russia,Izhevsk,3,Android,ads
3,2021-11-16 19:10:17,476,1234,view,0,19,1,11,1,23,Russia,Izhevsk,3,Android,ads
4,2021-11-16 19:13:15,476,1762,view,0,19,1,11,1,23,Russia,Izhevsk,3,Android,ads


In [46]:
final_table = pd.merge(
    learning_table, post_tfidf_processed_df, on="post_id", how="left"
)
print(final_table.shape)
final_table.head()

(4468243, 66)


,timestamp,user_id,post_id,action,target,hour,dow,month,gender,age,...,tfidf_think,tfidf_time,tfidf_try,tfidf_want,tfidf_watch,tfidf_way,tfidf_win,tfidf_work,tfidf_world,tfidf_year
0,2021-11-16 19:05:49,476,3873,view,1,19,1,11,1,23,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000
1,2021-11-16 19:06:36,476,720,view,0,19,1,11,1,23,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.220562,0.175437
2,2021-11-16 19:07:51,476,578,view,0,19,1,11,1,23,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.229439,0.0,0.000000,0.163300
3,2021-11-16 19:10:17,476,1234,view,0,19,1,11,1,23,...,0.245737,0.000000,0.058025,0.052161,0.0,0.0,0.000000,0.0,0.000000,0.000000
4,2021-11-16 19:13:15,476,1762,view,0,19,1,11,1,23,...,0.127328,0.110466,0.000000,0.405410,0.0,0.0,0.645295,0.0,0.000000,0.000000


In [47]:
print(final_table.shape)
final_table.head()

(4468243, 66)


,timestamp,user_id,post_id,action,target,hour,dow,month,gender,age,...,tfidf_think,tfidf_time,tfidf_try,tfidf_want,tfidf_watch,tfidf_way,tfidf_win,tfidf_work,tfidf_world,tfidf_year
0,2021-11-16 19:05:49,476,3873,view,1,19,1,11,1,23,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000
1,2021-11-16 19:06:36,476,720,view,0,19,1,11,1,23,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.220562,0.175437
2,2021-11-16 19:07:51,476,578,view,0,19,1,11,1,23,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.229439,0.0,0.000000,0.163300
3,2021-11-16 19:10:17,476,1234,view,0,19,1,11,1,23,...,0.245737,0.000000,0.058025,0.052161,0.0,0.0,0.000000,0.0,0.000000,0.000000
4,2021-11-16 19:13:15,476,1762,view,0,19,1,11,1,23,...,0.127328,0.110466,0.000000,0.405410,0.0,0.0,0.645295,0.0,0.000000,0.000000
